# Create Finska Läkaresällskapet Awards (Finnish Medical Society, FLS)

Creates Finska Läkaresällskapet research-grant awards from the society's own
"Beviljade forskningsanslag" page, https://fls.fi/beviljade-forskningsanslag/ ,
which is replaced every February by the new cohort. The live page gives the
2026-2027 cohort; earlier cohorts come from Internet Archive captures of the
same page: 2021 (20211208), 2022 (20220302), 2023-2024 (20230331), 2025-2026
(20250806). **577 awards, cohorts 2021-2026, EUR 13.5M** (local run 2026-10-01).
Not covered: the 2024-2025 cohort (Feb 2024 page never captured) and pre-2021
lists (PDFs on the old site, mostly not archived).

**Source format / checks:** per category, a recipient line and the euro amount
(two amounts = the two years of a 2-year grant; `amount` = their sum, raw text
in `amount_parts`). First-year sums reconcile with the total printed on each
page (2023 and 2026 exact; 2021/2022 differ by exactly the Einar och Karin
Stroems stiftelse grants plus, for 2022, one malformed amount '27 00', which
ships NULL). No project titles, institutions or grant numbers are published.

**Mapping:** `display_name` is composed as '<category>: <recipient> (<cohort>)'
(Sigrid Jusélius precedent). `funder_scheme` = the category (Äldre forskare,
Yngre forskare, Forskare som ej doktorerat, Doktorandstipendium,
(Postdoc-)forskning utomlands, and the FLS-administered Linda Gadds fond,
Einar och Karin Stroems stiftelse, Kurt och Doris Palanders stiftelse, Elli
Maria och Anna Sofia Ruuths fond). Doktorandstipendium lines read
'Supervisor / Doctoral student': the supervisor applies and is accountable
(FLS rules), so lead = supervisor, co_lead = doctoral student. Names are
'Given Family' up to 2023 and 'Family Given' from 2025 (split accordingly).
`start_date` = the disbursement date printed on the page ('för utdelning
6.2.2026'); `start_year` = cohort year; `end_year` = second cohort year for
2-year cohorts.

**`funder_award_id` (§2.1.1):** FLS publishes no grant number and citing works
write none consistently (the 75 distinct existing stubs are mostly other funders'
numbers or years); synthetic `FLS-<cohort year>-<category code>-<recipient slug>`.
0 stubs collapse. research.fi does not carry FLS.

**Prerequisites:** run `scripts/local/finska_lakaresallskapet_to_s3.py` first. It uploads
`s3://openalex-ingest/awards/finska_lakaresallskapet/finska_lakaresallskapet_projects.parquet`
(§1.4 shrink guard).

**Funder (Path A, F4320* Crossref-registered):** funder_id `4320311816`,
Finska Läkaresällskapet (ror 04vbzm272, doi 10.13039/100010135), read from
`openalex.funders.funders` by id. No twin row.

**Priority:** `582` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.finska_lakaresallskapet_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/finska_lakaresallskapet/finska_lakaresallskapet_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_rows, COUNT(DISTINCT funder_award_id) as distinct_ids
FROM openalex.awards.finska_lakaresallskapet_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.finska_lakaresallskapet_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.finska_lakaresallskapet_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320311816 must resolve to exactly 1 row in `openalex.funders.funders`. If 0 rows, STOP and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320311816;

## Step 2: Create Finska Läkaresällskapet Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.finska_lakaresallskapet_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320311816  -- Finska Läkaresällskapet
),
src AS (
    SELECT
        TRIM(funder_award_id) AS funder_award_id,
        NULLIF(TRIM(funder_scheme), '') AS funder_scheme,
        NULLIF(TRIM(funding_type), '') AS funding_type,
        NULLIF(TRIM(recipient_raw), '') AS recipient,
        TRY_CAST(cohort_start_year AS INT) AS y0,
        TRY_CAST(cohort_end_year AS INT) AS y1,
        TRY_TO_DATE(decision_date, 'yyyy-MM-dd') AS decision_date,
        TRY_CAST(amount AS DOUBLE) AS amount,
        NULLIF(TRIM(lead_given_name), '') AS given_name,
        NULLIF(TRIM(lead_family_name), '') AS family_name,
        NULLIF(TRIM(second_given_name), '') AS co_given_name,
        NULLIF(TRIM(second_family_name), '') AS co_family_name,
        source_url
    FROM openalex.awards.finska_lakaresallskapet_raw
    WHERE funder_award_id IS NOT NULL AND TRIM(funder_award_id) != ''
)
SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 as id,
    CONCAT(g.funder_scheme, ': ', g.recipient, ' (',
           CAST(g.y0 AS STRING), CASE WHEN g.y1 IS NOT NULL THEN CONCAT('-', CAST(g.y1 AS STRING)) ELSE '' END,
           ')') as display_name,
    CAST(NULL AS STRING) as description,
    f.funder_id,
    g.funder_award_id,
    g.amount,
    CASE WHEN g.amount IS NOT NULL THEN 'EUR' END as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    COALESCE(g.funding_type, 'research') as funding_type,
    g.funder_scheme,
    'fls_beviljade_forskningsanslag' as provenance,
    g.decision_date as start_date,
    CAST(NULL AS DATE) as end_date,
    g.y0 as start_year,
    g.y1 as end_year,
    CASE WHEN g.family_name IS NOT NULL THEN named_struct(
        'given_name', g.given_name,
        'family_name', g.family_name,
        'orcid', CAST(NULL AS STRING),
        'role_start', CAST(NULL AS DATE),
        'affiliation', CAST(NULL AS STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>)
    ) END as lead_investigator,
    CASE WHEN g.co_family_name IS NOT NULL THEN named_struct(
        'given_name', g.co_given_name,
        'family_name', g.co_family_name,
        'orcid', CAST(NULL AS STRING),
        'role_start', CAST(NULL AS DATE),
        'affiliation', CAST(NULL AS STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>)
    ) END as co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>>) as investigators,
    g.source_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM src g
CROSS JOIN src_funder f;

In [ ]:
%sql
-- §2.1.1 shape check: synthetic FLS- keys only; expect 0 rows.
SELECT funder_award_id FROM openalex.awards.finska_lakaresallskapet_awards
WHERE NOT funder_award_id RLIKE '^FLS-20[0-9]{2}-[A-Z]{2}-[a-z0-9-]+$'
LIMIT 20;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'fls_beviljade_forskningsanslag' AND priority = 582;

-- Priority 582: direct-from-funder ingest of FLS's grant lists.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    582 as priority
FROM openalex.awards.finska_lakaresallskapet_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_awards, COUNT(DISTINCT id) as distinct_ids
FROM openalex.awards.finska_lakaresallskapet_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, amount, currency, start_date, start_year, end_year,
       lead_investigator.given_name, lead_investigator.family_name,
       co_lead_investigator.given_name AS co_given, co_lead_investigator.family_name AS co_family
FROM openalex.awards.finska_lakaresallskapet_awards LIMIT 20;

In [ ]:
%sql
SELECT start_year, funder_scheme, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_eur
FROM openalex.awards.finska_lakaresallskapet_awards
GROUP BY start_year, funder_scheme ORDER BY start_year, cnt DESC;

In [ ]:
%sql
-- Section 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.finska_lakaresallskapet_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.7 coverage (expect 100% title/PI/start_date, ~99.8% amount).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(amount) as has_amount,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) as pct_amount,
    collect_set(currency) as currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    ROUND(AVG(amount), 0) as avg_amt,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(co_lead_investigator.family_name) as has_co_lead
FROM openalex.awards.finska_lakaresallskapet_awards;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'fls_beviljade_forskningsanslag'
GROUP BY provenance, priority;